# Day 02 — Overfitting, Underfitting & Validation 🧪

This is an **optional starter lab** to test ideas from Day 02. We create a small, noisy dataset in Python: no data cleaning or external files.

We will:
1. Fit models of different complexity.
2. Compare training and validation error.
3. Choose model complexity using **5-fold cross-validation**.
4. Open the held-out test set only after model selection.

**MSE (mean squared error)**: the average squared prediction error. Lower is better.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(42)
X = rng.uniform(0, 1, size=(80, 1))
y = np.sin(2 * np.pi * X[:, 0]) + rng.normal(0, 0.18, size=80)

# Reserve the final test examples before any model selection.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42
)
print("Training:", len(X_train), "Validation:", len(X_val), "Final test:", len(X_test))

## Part A — Compare model complexity

- **Degree 1:** a straight line (may underfit curved data).
- **Degree 5:** a more flexible curve.
- **Degree 18:** a very flexible curve (may overfit noise).

Observe that a lower training error does **not** always mean better validation performance.

In [ ]:
degrees = [1, 5, 18]
x_grid = np.linspace(0, 1, 250).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, degree in zip(axes, degrees):
    candidate = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    candidate.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, candidate.predict(X_train))
    val_mse = mean_squared_error(y_val, candidate.predict(X_val))
    print(f"Degree {degree:>2}: train MSE={train_mse:.3f}, validation MSE={val_mse:.3f}")

    ax.scatter(X_train[:, 0], y_train, s=17, label="Train samples")
    ax.scatter(X_val[:, 0], y_val, s=17, marker="x", label="Validation samples")
    ax.plot(x_grid[:, 0], candidate.predict(x_grid), linewidth=2)
    ax.set_title(f"Degree {degree}")
    ax.set_xlabel("X")
    ax.set_ylim(-1.8, 1.8)
axes[0].set_ylabel("y")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Reflect:** Which degree has the lowest training error? Does the same degree also have the lowest validation error? This illustrates the importance of checking generalization.

## Part B — Use cross-validation to select the degree

We perform 5-fold CV **only on development data**. Each fold takes a turn being the validation fold. There is no use of the held-out test data during the comparison.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
candidate_degrees = [1, 3, 5, 9, 18]
cv_results = {}

for degree in candidate_degrees:
    candidate = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )
    neg_mse_scores = cross_val_score(
        candidate, X_dev, y_dev, cv=cv, scoring="neg_mean_squared_error"
    )
    cv_results[degree] = -neg_mse_scores.mean()  # Convert negative MSE to positive MSE
    print(f"Degree {degree:>2}: mean CV MSE = {cv_results[degree]:.3f}")

best_degree = min(cv_results, key=cv_results.get)
print("Selected degree from CV:", best_degree)

## Part C — Final evaluation (only now)

After choosing the degree using CV, retrain on **all development data** and evaluate once on the held-out test set. We should not tune the model based on this result.

In [ ]:
final_model = make_pipeline(
    PolynomialFeatures(degree=best_degree, include_bias=False),
    LinearRegression(),
)
final_model.fit(X_dev, y_dev)
final_test_mse = mean_squared_error(y_test, final_model.predict(X_test))
print(f"Final held-out test MSE: {final_test_mse:.3f}")

## Your experiment

- Try adding degree `2` or `7` to `candidate_degrees`.
- Compare the scores and explain whether more complexity always helped.
- Explain why you should **not** choose a different degree just because its test score is better.

**My observation (write after running):**

- What I expected:
- What I saw:
- What this teaches me: